# Chuyển `customer_turns` thành audio bằng ElevenLabs TTS

Notebook Google Colab này đọc **một file scenario JSON**, lấy `calls.<call_id>.customer_turns`, rồi tạo **một file audio cho mỗi cuộc gọi** bằng ElevenLabs Text-to-Speech API.

Trước khi chạy:

1. Chạy ô cài đặt thư viện và mount Google Drive.
2. Lưu API key với tên `ELEVENLABS_API_KEY` trong **Colab Secrets**, hoặc thêm `ELEVENLABS_API_KEY=...` vào `/content/.env`. Không ghi key trực tiếp vào notebook.
3. Chọn `SCENARIO_FILE`, kiểm tra phần xem trước, rồi đổi `RUN_TTS = True`.
4. Có thể thay `VOICE_ID` bằng voice ID trong tài khoản ElevenLabs của bạn.

> Mặc định `RUN_TTS = False` để Run All không vô tình sử dụng quota. API tham chiếu: https://elevenlabs.io/docs/api-reference/text-to-speech/convert

In [ ]:
%pip install -q requests python-dotenv

In [ ]:
import json
import os
import re
from pathlib import Path

import requests
from dotenv import load_dotenv
from IPython.display import Audio, display

## 1. Mount Google Drive và chọn file đầu vào

Đổi `SCENARIO_FILE` nếu file của bạn có tên hoặc vị trí khác. Các đường dẫn phải là đối tượng `Path`.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content")
DATASET_DIR = Path("/content/drive/MyDrive/scenarios_complete dataset")

# Thay tên file tại đây nếu cần.
SCENARIO_FILE = DATASET_DIR / "SC-001.json"
OUTPUT_DIR = Path("/content")

if not SCENARIO_FILE.is_file():
    raise FileNotFoundError(f"Không tìm thấy file: {SCENARIO_FILE}")

print("Input :", SCENARIO_FILE)
print("Output:", OUTPUT_DIR)

## 2. Đọc và kiểm tra `customer_turns`

In [ ]:
def load_customer_turns(json_path: Path) -> tuple[str, dict[str, list[str]]]:
    """Đọc scenario và trả về (scenario_id, {call_id: customer_turns})."""
    with json_path.open("r", encoding="utf-8") as file:
        payload = json.load(file)

    if not isinstance(payload, dict):
        raise ValueError("Nội dung JSON phải là một object.")

    scenario_id = str(payload.get("scenario_id") or json_path.stem)
    calls = payload.get("calls")
    if not isinstance(calls, dict) or not calls:
        raise ValueError("Không tìm thấy object 'calls' hoặc 'calls' đang rỗng.")

    turns_by_call: dict[str, list[str]] = {}
    for call_id, call_data in calls.items():
        if not isinstance(call_data, dict):
            raise ValueError(f"{call_id} phải là một object.")

        turns = call_data.get("customer_turns")
        if not isinstance(turns, list):
            raise ValueError(f"{call_id}.customer_turns phải là một list.")
        if not all(isinstance(turn, str) for turn in turns):
            raise ValueError(f"Mọi phần tử trong {call_id}.customer_turns phải là chuỗi.")

        cleaned_turns = [turn.strip() for turn in turns if turn.strip()]
        if not cleaned_turns:
            raise ValueError(f"{call_id}.customer_turns không có câu hợp lệ.")
        turns_by_call[str(call_id)] = cleaned_turns

    return scenario_id, turns_by_call


scenario_id, customer_turns_by_call = load_customer_turns(SCENARIO_FILE)
print(f"Scenario: {scenario_id}")
for call_id, turns in customer_turns_by_call.items():
    print(f"\n{call_id} ({len(turns)} câu):")
    for index, text in enumerate(turns, start=1):
        print(f"  {index}. {text}")

## 3. Cấu hình ElevenLabs TTS

- `CALL_IDS = None`: xử lý tất cả các cuộc gọi; ví dụ `["call_1"]` để chỉ xử lý một call.
- `eleven_v3` hỗ trợ tiếng Việt và giới hạn 5.000 ký tự/request.
- Voice mặc định là George từ quickstart chính thức; hãy thay `VOICE_ID` nếu bạn muốn giọng khác.
- Mặc định xuất MP3 vì WAV 44.1 kHz có thể yêu cầu gói Pro.
- `OVERWRITE = False`: bỏ qua file đã tồn tại.

In [ ]:
API_BASE_URL = "https://api.elevenlabs.io/v1/text-to-speech"
MODEL_ID = "eleven_v3"
VOICE_ID = "JBFqnCBsd6RMkjVDRZzb"  # George; có thể thay bằng voice ID của bạn.
LANGUAGE_CODE = "vi"
OUTPUT_FORMAT = "mp3_44100_128"
APPLY_TEXT_NORMALIZATION = "auto"

STABILITY = 0.5
SIMILARITY_BOOST = 0.75
STYLE = 0.0
USE_SPEAKER_BOOST = True
SPEED = 1.0
REQUEST_TIMEOUT_SECONDS = 180

CALL_IDS = None  # None = tất cả; ví dụ: ["call_1"]
OVERWRITE = False
RUN_TTS = False  # Đổi thành True sau khi đã kiểm tra nội dung phía trên.

load_dotenv(PROJECT_ROOT / ".env")
ELEVENLABS_API_KEY = os.getenv("ELEVENLABS_API_KEY")

# Nếu chưa có trong biến môi trường/.env, thử lấy từ Colab Secrets.
if not ELEVENLABS_API_KEY:
    try:
        from google.colab import userdata

        ELEVENLABS_API_KEY = userdata.get("ELEVENLABS_API_KEY")
    except Exception:
        ELEVENLABS_API_KEY = None

print("Đã tìm thấy ELEVENLABS_API_KEY:", bool(ELEVENLABS_API_KEY))

In [ ]:
def safe_filename(value: str) -> str:
    return re.sub(r"[^A-Za-z0-9._-]+", "_", value).strip("._") or "audio"


def audio_extension(output_format: str) -> str:
    codec = output_format.split("_", maxsplit=1)[0].lower()
    known_extensions = {
        "mp3": "mp3",
        "wav": "wav",
        "pcm": "pcm",
        "opus": "opus",
        "ulaw": "ulaw",
        "alaw": "alaw",
    }
    if codec not in known_extensions:
        raise ValueError(f"Không nhận diện được OUTPUT_FORMAT: {output_format}")
    return known_extensions[codec]


def build_call_text(text_chunks: list[str]) -> str:
    # Hai dòng trống tạo ranh giới tự nhiên giữa các lượt nói của khách.
    text = "\n\n".join(chunk.strip() for chunk in text_chunks if chunk.strip())
    if not text:
        raise ValueError("Không có nội dung hợp lệ để tạo audio.")
    if len(text) > 5_000:
        raise ValueError(
            f"Nội dung có {len(text):,} ký tự; eleven_v3 giới hạn 5.000 ký tự/request."
        )
    return text


def synthesize_call(
    api_key: str,
    scenario_id: str,
    call_id: str,
    text_chunks: list[str],
    output_dir: Path,
) -> tuple[Path, int, dict]:
    """Gọi ElevenLabs TTS và tạo một file audio cho một call."""
    text = build_call_text(text_chunks)
    extension = audio_extension(OUTPUT_FORMAT)
    url = f"{API_BASE_URL}/{VOICE_ID}"
    params = {"output_format": OUTPUT_FORMAT}
    headers = {
        "xi-api-key": api_key,
        "Content-Type": "application/json",
        "Accept": "audio/mpeg" if extension == "mp3" else "audio/*",
    }
    payload = {
        "text": text,
        "model_id": MODEL_ID,
        "language_code": LANGUAGE_CODE,
        "apply_text_normalization": APPLY_TEXT_NORMALIZATION,
        "voice_settings": {
            "stability": STABILITY,
            "similarity_boost": SIMILARITY_BOOST,
            "style": STYLE,
            "use_speaker_boost": USE_SPEAKER_BOOST,
            "speed": SPEED,
        },
    }

    response = requests.post(
        url,
        params=params,
        headers=headers,
        json=payload,
        timeout=REQUEST_TIMEOUT_SECONDS,
    )
    if not response.ok:
        try:
            detail = response.json()
        except ValueError:
            detail = response.text[:1_000]
        raise RuntimeError(f"ElevenLabs HTTP {response.status_code}: {detail}")

    content_type = response.headers.get("content-type", "")
    if "application/json" in content_type.lower():
        raise RuntimeError(f"ElevenLabs trả JSON thay vì audio: {response.text[:1_000]}")

    audio = response.content
    if not audio:
        raise RuntimeError("ElevenLabs trả về audio rỗng.")

    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / (
        f"{safe_filename(scenario_id)}_{safe_filename(call_id)}_elevenlabs_customer.{extension}"
    )
    output_path.write_bytes(audio)

    metadata = {
        "request_id": response.headers.get("request-id"),
        "history_item_id": response.headers.get("history-item-id"),
        "character_cost": response.headers.get("character-cost"),
        "text_characters": len(text),
        "content_type": content_type,
    }
    return output_path, len(audio), metadata

## 4. Gọi ElevenLabs và lưu audio

Ô này chỉ gọi API khi `RUN_TTS = True`. Kết quả được lưu trong `/content/<scenario_id>/`, kèm `manifest_elevenlabs.json`.

In [ ]:
if not RUN_TTS:
    print("Chưa gọi API. Hãy đổi RUN_TTS = True trong ô cấu hình khi sẵn sàng.")
else:
    if not ELEVENLABS_API_KEY:
        raise EnvironmentError(
            "Thiếu ELEVENLABS_API_KEY trong Colab Secrets, /content/.env hoặc biến môi trường."
        )

    if CALL_IDS is None:
        selected_call_ids = list(customer_turns_by_call)
    else:
        if isinstance(CALL_IDS, str):
            raise TypeError("CALL_IDS phải là list, ví dụ ['call_1'], không phải chuỗi.")
        selected_call_ids = list(dict.fromkeys(CALL_IDS))
        missing = [call_id for call_id in selected_call_ids if call_id not in customer_turns_by_call]
        if missing:
            raise KeyError(f"CALL_IDS không tồn tại trong file: {missing}")

    extension = audio_extension(OUTPUT_FORMAT)
    scenario_output_dir = OUTPUT_DIR / safe_filename(scenario_id)
    results = []
    generated_files = []

    for call_id in selected_call_ids:
        expected_path = scenario_output_dir / (
            f"{safe_filename(scenario_id)}_{safe_filename(call_id)}_elevenlabs_customer.{extension}"
        )
        if expected_path.exists() and not OVERWRITE:
            print(f"Bỏ qua file đã tồn tại: {expected_path.name}")
            results.append(
                {
                    "call_id": call_id,
                    "status": "skipped_existing",
                    "output_file": str(expected_path),
                }
            )
            continue

        try:
            output_path, byte_count, metadata = synthesize_call(
                api_key=ELEVENLABS_API_KEY,
                scenario_id=scenario_id,
                call_id=call_id,
                text_chunks=customer_turns_by_call[call_id],
                output_dir=scenario_output_dir,
            )
            generated_files.append(output_path)
            results.append(
                {
                    "call_id": call_id,
                    "status": "generated",
                    "turn_count": len(customer_turns_by_call[call_id]),
                    "bytes": byte_count,
                    "output_file": str(output_path),
                    **metadata,
                }
            )
            print(f"Đã ghi {byte_count:,} bytes: {output_path}")
        except Exception as exc:
            results.append(
                {
                    "call_id": call_id,
                    "status": "error",
                    "error": f"{type(exc).__name__}: {exc}",
                }
            )
            print(f"Lỗi ở {call_id}: {type(exc).__name__}: {exc}")

    scenario_output_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = scenario_output_dir / "manifest_elevenlabs.json"
    manifest = {
        "source_file": str(SCENARIO_FILE),
        "scenario_id": scenario_id,
        "model_id": MODEL_ID,
        "voice_id": VOICE_ID,
        "language_code": LANGUAGE_CODE,
        "output_format": OUTPUT_FORMAT,
        "results": results,
    }
    manifest_path.write_text(
        json.dumps(manifest, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    print("Manifest:", manifest_path)

    if generated_files:
        print("Nghe thử file vừa tạo đầu tiên:")
        display(Audio(filename=str(generated_files[0])))